# Nail U-Net Training for Google Colab

This notebook trains the binary fingernail segmentation model used by the robot runtime in `nailbot/vision.py`.

It is the canonical training path for this repo. The important design rule is:

**The notebook U-Net, preprocessing, image size, and exported checkpoint must match the Raspberry Pi runtime.**

Expected dataset layout in Google Drive:

```text
/content/drive/MyDrive/NailSegmentationDatasetV2/
  train/images/
  train/masks/
  val/images/
  val/masks/
  test/images/
  test/masks/
```

Main outputs:

- `nail_unet_best.pt`: best validation checkpoint
- `nail_unet_last.pt`: last epoch checkpoint
- `nail_unet.pt`: deploy-ready checkpoint to copy locally into `models/nail_unet.pt`
- `nail_unet_pi_state_dict.pt`: plain state dict fallback
- `nail_unet_metadata.json`: threshold, image size, normalization, and final metrics
- `training_history.csv`: epoch metrics
- `prediction_quality_panel.png`: visual sanity check

## 1. Runtime Setup

In [ ]:
!pip install -q torch torchvision matplotlib pillow tqdm pandas scipy

from google.colab import drive
drive.mount('/content/drive')

## 2. Paths, Hyperparameters, and Reproducibility

In [ ]:
from pathlib import Path
import json
import random

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import matplotlib.pyplot as plt
from PIL import Image
from scipy import ndimage as ndi
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from torchvision.transforms import InterpolationMode
import torchvision.transforms.functional as TF
from tqdm.auto import tqdm

DATASET_ROOT = Path('/content/drive/MyDrive/NailSegmentationDatasetV2')
OUTPUT_DIR = Path('/content/drive/MyDrive/nail_unet_outputs')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Keep this aligned with nailbot.config.VisionConfig.image_size.
IMAGE_SIZE = (512, 512)
BATCH_SIZE = 8
NUM_EPOCHS = 40
LEARNING_RATE = 1e-3
WEIGHT_DECAY = 1e-4
NUM_WORKERS = 2
SEED = 42

BEST_MODEL_PATH = OUTPUT_DIR / 'nail_unet_best.pt'
LAST_MODEL_PATH = OUTPUT_DIR / 'nail_unet_last.pt'
DEPLOY_MODEL_PATH = OUTPUT_DIR / 'nail_unet.pt'
PLAIN_STATE_DICT_PATH = OUTPUT_DIR / 'nail_unet_pi_state_dict.pt'
METADATA_PATH = OUTPUT_DIR / 'nail_unet_metadata.json'
HISTORY_CSV_PATH = OUTPUT_DIR / 'training_history.csv'
PREDICTION_PANEL_PATH = OUTPUT_DIR / 'prediction_quality_panel.png'

IMAGE_MEAN = [0.485, 0.456, 0.406]
IMAGE_STD = [0.229, 0.224, 0.225]
INITIAL_THRESHOLD = 0.5

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.benchmark = True

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Using device:', device)
print('Dataset root:', DATASET_ROOT)
print('Output directory:', OUTPUT_DIR)

for split in ['train', 'val', 'test']:
    print(split, (DATASET_ROOT / split / 'images').exists(), (DATASET_ROOT / split / 'masks').exists())

## 3. Dataset and Dataloaders

In [ ]:
IMAGE_EXTENSIONS = {'.jpg', '.jpeg', '.png', '.bmp'}
MASK_EXTENSIONS = {'.png', '.jpg', '.jpeg', '.bmp'}


def find_mask_for_image(mask_dir: Path, image_path: Path) -> Path | None:
    for extension in MASK_EXTENSIONS:
        mask_path = mask_dir / f'{image_path.stem}{extension}'
        if mask_path.exists():
            return mask_path
    return None


class NailSegmentationDataset(Dataset):
    def __init__(self, root: Path, split: str, image_size=(512, 512), augment=False):
        self.root = Path(root)
        self.split = split
        self.images_dir = self.root / split / 'images'
        self.masks_dir = self.root / split / 'masks'
        self.image_size = image_size
        self.augment = augment

        if not self.images_dir.exists() or not self.masks_dir.exists():
            raise FileNotFoundError(f'Missing split folders: {self.images_dir} and/or {self.masks_dir}')

        self.samples = []
        for image_path in sorted(self.images_dir.iterdir()):
            if image_path.suffix.lower() not in IMAGE_EXTENSIONS:
                continue
            mask_path = find_mask_for_image(self.masks_dir, image_path)
            if mask_path is not None:
                self.samples.append((image_path, mask_path))

        if not self.samples:
            raise RuntimeError(f'No image-mask pairs found in {self.images_dir} and {self.masks_dir}')

        self.normalize = transforms.Normalize(mean=IMAGE_MEAN, std=IMAGE_STD)

    def __len__(self):
        return len(self.samples)

    def _resize_pair(self, image, mask):
        image = TF.resize(image, self.image_size, interpolation=InterpolationMode.BILINEAR)
        mask = TF.resize(mask, self.image_size, interpolation=InterpolationMode.NEAREST)
        return image, mask

    def _augment_pair(self, image, mask):
        if random.random() < 0.5:
            image = TF.hflip(image)
            mask = TF.hflip(mask)
        if random.random() < 0.25:
            image = TF.vflip(image)
            mask = TF.vflip(mask)

        angle = random.uniform(-18, 18)
        translate = (random.uniform(-0.03, 0.03) * self.image_size[1], random.uniform(-0.03, 0.03) * self.image_size[0])
        scale = random.uniform(0.92, 1.08)
        image = TF.affine(
            image,
            angle=angle,
            translate=translate,
            scale=scale,
            shear=[0.0, 0.0],
            interpolation=InterpolationMode.BILINEAR,
            fill=0,
        )
        mask = TF.affine(
            mask,
            angle=angle,
            translate=translate,
            scale=scale,
            shear=[0.0, 0.0],
            interpolation=InterpolationMode.NEAREST,
            fill=0,
        )

        if random.random() < 0.8:
            jitter = transforms.ColorJitter(brightness=0.20, contrast=0.20, saturation=0.15, hue=0.02)
            image = jitter(image)
        return image, mask

    def __getitem__(self, index):
        image_path, mask_path = self.samples[index]
        image = Image.open(image_path).convert('RGB')
        mask = Image.open(mask_path).convert('L')

        image, mask = self._resize_pair(image, mask)
        if self.augment:
            image, mask = self._augment_pair(image, mask)

        image_tensor = TF.to_tensor(image)
        image_tensor = self.normalize(image_tensor)
        mask_tensor = TF.to_tensor(mask)
        mask_tensor = (mask_tensor > 0.5).float()
        return image_tensor, mask_tensor, image_path.name


def make_loader(dataset, shuffle):
    return DataLoader(
        dataset,
        batch_size=BATCH_SIZE,
        shuffle=shuffle,
        num_workers=NUM_WORKERS,
        pin_memory=(device.type == 'cuda'),
    )


train_dataset = NailSegmentationDataset(DATASET_ROOT, 'train', image_size=IMAGE_SIZE, augment=True)
val_dataset = NailSegmentationDataset(DATASET_ROOT, 'val', image_size=IMAGE_SIZE, augment=False)
test_dataset = NailSegmentationDataset(DATASET_ROOT, 'test', image_size=IMAGE_SIZE, augment=False)

train_loader = make_loader(train_dataset, shuffle=True)
val_loader = make_loader(val_dataset, shuffle=False)
test_loader = make_loader(test_dataset, shuffle=False)

print('Train:', len(train_dataset))
print('Val:', len(val_dataset))
print('Test:', len(test_dataset))
print('Example:', train_dataset.samples[0])

## 4. Visual Check: Images and Masks

In [ ]:
def denormalize_image(image_tensor):
    mean = torch.tensor(IMAGE_MEAN).view(3, 1, 1)
    std = torch.tensor(IMAGE_STD).view(3, 1, 1)
    image = image_tensor.detach().cpu() * std + mean
    return image.clamp(0, 1)


images, masks, names = next(iter(train_loader))
show_count = min(4, images.size(0))
fig, axes = plt.subplots(show_count, 2, figsize=(8, show_count * 3))
if show_count == 1:
    axes = np.array([axes])
for row in range(show_count):
    axes[row, 0].imshow(denormalize_image(images[row]).permute(1, 2, 0).numpy())
    axes[row, 0].set_title(names[row])
    axes[row, 0].axis('off')
    axes[row, 1].imshow(masks[row, 0].numpy(), cmap='gray')
    axes[row, 1].set_title('mask')
    axes[row, 1].axis('off')
plt.tight_layout()
plt.show()

## 5. Runtime-Compatible U-Net Model

In [ ]:
class TwoConvLayers(nn.Module):
    def __init__(self, in_channels: int, out_channels: int):
        super().__init__()
        self.model = nn.Sequential(
            nn.Conv2d(in_channels, out_channels, kernel_size=3, padding=1, bias=False),
            nn.ReLU(inplace=True),
            nn.BatchNorm2d(out_channels),
            nn.Conv2d(out_channels, out_channels, kernel_size=3, padding=1, bias=False),
            nn.ReLU(inplace=True),
            nn.BatchNorm2d(out_channels),
        )

    def forward(self, x):
        return self.model(x)


class Encoder(nn.Module):
    def __init__(self, in_channels: int, out_channels: int):
        super().__init__()
        self.block = TwoConvLayers(in_channels, out_channels)
        self.max_pool = nn.MaxPool2d(kernel_size=2, stride=2)

    def forward(self, x):
        skip = self.block(x)
        return self.max_pool(skip), skip


class Decoder(nn.Module):
    def __init__(self, in_channels: int, out_channels: int):
        super().__init__()
        self.transpose = nn.ConvTranspose2d(in_channels, out_channels, kernel_size=2, stride=2)
        self.block = TwoConvLayers(in_channels, out_channels)

    def forward(self, x, skip):
        x = self.transpose(x)
        x = torch.cat([x, skip], dim=1)
        return self.block(x)


class UNet(nn.Module):
    def __init__(self, in_channels: int = 3, num_classes: int = 1):
        super().__init__()
        self.enc_block1 = Encoder(in_channels, 32)
        self.enc_block2 = Encoder(32, 64)
        self.enc_block3 = Encoder(64, 128)
        self.enc_block4 = Encoder(128, 256)
        self.bottleneck = TwoConvLayers(256, 512)
        self.dec_block1 = Decoder(512, 256)
        self.dec_block2 = Decoder(256, 128)
        self.dec_block3 = Decoder(128, 64)
        self.dec_block4 = Decoder(64, 32)
        self.output = nn.Conv2d(32, num_classes, kernel_size=1)

    def forward(self, x):
        x, skip1 = self.enc_block1(x)
        x, skip2 = self.enc_block2(x)
        x, skip3 = self.enc_block3(x)
        x, skip4 = self.enc_block4(x)
        x = self.bottleneck(x)
        x = self.dec_block1(x, skip4)
        x = self.dec_block2(x, skip3)
        x = self.dec_block3(x, skip2)
        x = self.dec_block4(x, skip1)
        return self.output(x)


model = UNet().to(device)
with torch.no_grad():
    test_logits = model(torch.zeros(1, 3, *IMAGE_SIZE, device=device))
print('Output shape:', tuple(test_logits.shape))
assert tuple(test_logits.shape) == (1, 1, IMAGE_SIZE[0], IMAGE_SIZE[1])

## 6. Compute Foreground Class Weight

In [ ]:
def compute_positive_weight(dataset, max_samples=None):
    positive_pixels = 0
    total_pixels = 0
    sample_count = len(dataset) if max_samples is None else min(len(dataset), max_samples)
    for index in tqdm(range(sample_count), desc='Scanning masks'):
        _, mask_path = dataset.samples[index]
        mask = Image.open(mask_path).convert('L')
        mask = TF.resize(mask, IMAGE_SIZE, interpolation=InterpolationMode.NEAREST)
        mask_np = np.array(mask) > 127
        positive_pixels += int(mask_np.sum())
        total_pixels += int(mask_np.size)

    negative_pixels = total_pixels - positive_pixels
    raw_weight = negative_pixels / max(positive_pixels, 1)
    # Avoid extreme weights that can make the model paint the whole image or nothing at all.
    clipped_weight = float(np.clip(raw_weight, 1.0, 20.0))
    foreground_pct = positive_pixels / max(total_pixels, 1) * 100
    return clipped_weight, foreground_pct


POS_WEIGHT, FOREGROUND_PCT = compute_positive_weight(train_dataset)
print(f'Foreground pixels: {FOREGROUND_PCT:.2f}%')
print(f'Using BCE pos_weight: {POS_WEIGHT:.3f}')

## 7. Loss, Metrics, and Training Helpers

In [ ]:
class DiceLoss(nn.Module):
    def __init__(self, smooth=1.0):
        super().__init__()
        self.smooth = smooth

    def forward(self, logits, targets):
        probs = torch.sigmoid(logits)
        probs = probs.flatten(1)
        targets = targets.flatten(1)
        intersection = (probs * targets).sum(dim=1)
        denominator = probs.sum(dim=1) + targets.sum(dim=1)
        dice = (2.0 * intersection + self.smooth) / (denominator + self.smooth)
        return 1.0 - dice.mean()


def dice_score_from_probs(probs, targets, threshold=0.5, smooth=1.0):
    preds = (probs > threshold).float().flatten(1)
    targets = targets.flatten(1)
    intersection = (preds * targets).sum(dim=1)
    denominator = preds.sum(dim=1) + targets.sum(dim=1)
    dice = (2.0 * intersection + smooth) / (denominator + smooth)
    return dice.mean().item()


def iou_score_from_probs(probs, targets, threshold=0.5, smooth=1.0):
    preds = (probs > threshold).float().flatten(1)
    targets = targets.flatten(1)
    intersection = (preds * targets).sum(dim=1)
    union = preds.sum(dim=1) + targets.sum(dim=1) - intersection
    iou = (intersection + smooth) / (union + smooth)
    return iou.mean().item()


bce_loss = nn.BCEWithLogitsLoss(pos_weight=torch.tensor([POS_WEIGHT], device=device))
dice_loss = DiceLoss()


def segmentation_loss(logits, targets):
    return 0.45 * bce_loss(logits, targets) + 0.55 * dice_loss(logits, targets)


def run_epoch(model, loader, optimizer=None, threshold=INITIAL_THRESHOLD):
    is_train = optimizer is not None
    model.train(is_train)
    total_loss = 0.0
    total_dice = 0.0
    total_iou = 0.0

    autocast_enabled = device.type == 'cuda'
    scaler = torch.cuda.amp.GradScaler(enabled=autocast_enabled and is_train)

    progress = tqdm(loader, leave=False)
    for images, masks, _ in progress:
        images = images.to(device, non_blocking=True)
        masks = masks.to(device, non_blocking=True)

        if is_train:
            optimizer.zero_grad(set_to_none=True)

        with torch.set_grad_enabled(is_train):
            with torch.cuda.amp.autocast(enabled=autocast_enabled):
                logits = model(images)
                loss = segmentation_loss(logits, masks)

            if is_train:
                scaler.scale(loss).backward()
                scaler.step(optimizer)
                scaler.update()

        probs = torch.sigmoid(logits.detach())
        batch_dice = dice_score_from_probs(probs, masks, threshold=threshold)
        batch_iou = iou_score_from_probs(probs, masks, threshold=threshold)
        total_loss += loss.item() * images.size(0)
        total_dice += batch_dice * images.size(0)
        total_iou += batch_iou * images.size(0)
        progress.set_postfix(loss=f'{loss.item():.4f}', dice=f'{batch_dice:.4f}', iou=f'{batch_iou:.4f}')

    dataset_size = len(loader.dataset)
    return total_loss / dataset_size, total_dice / dataset_size, total_iou / dataset_size

## 8. Train the Model

In [ ]:
optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='max', factor=0.5, patience=3)

history = []
best_val_dice = -1.0
best_epoch = None

for epoch in range(1, NUM_EPOCHS + 1):
    train_loss, train_dice, train_iou = run_epoch(model, train_loader, optimizer=optimizer)
    val_loss, val_dice, val_iou = run_epoch(model, val_loader, optimizer=None)
    scheduler.step(val_dice)

    record = {
        'epoch': epoch,
        'train_loss': train_loss,
        'train_dice_at_0_5': train_dice,
        'train_iou_at_0_5': train_iou,
        'val_loss': val_loss,
        'val_dice_at_0_5': val_dice,
        'val_iou_at_0_5': val_iou,
        'lr': optimizer.param_groups[0]['lr'],
    }
    history.append(record)
    print(record)

    checkpoint = {
        'model_state_dict': model.state_dict(),
        'history': history,
        'image_size': IMAGE_SIZE,
        'threshold': INITIAL_THRESHOLD,
        'normalization': {'mean': IMAGE_MEAN, 'std': IMAGE_STD},
        'architecture': 'runtime_unet_v1',
        'pos_weight': POS_WEIGHT,
        'foreground_pct': FOREGROUND_PCT,
        'epoch': epoch,
    }
    torch.save(checkpoint, LAST_MODEL_PATH)

    if val_dice > best_val_dice:
        best_val_dice = val_dice
        best_epoch = epoch
        torch.save(checkpoint, BEST_MODEL_PATH)
        print(f'Saved new best model to {BEST_MODEL_PATH}')

history_df = pd.DataFrame(history)
history_df.to_csv(HISTORY_CSV_PATH, index=False)
print('Best validation Dice at 0.5:', best_val_dice, 'epoch:', best_epoch)
print('Saved history:', HISTORY_CSV_PATH)

## 9. Training Curves

In [ ]:
display(history_df.tail())

fig, axes = plt.subplots(1, 3, figsize=(16, 4))
axes[0].plot(history_df['epoch'], history_df['train_loss'], label='train')
axes[0].plot(history_df['epoch'], history_df['val_loss'], label='val')
axes[0].set_title('Loss')
axes[0].legend()

axes[1].plot(history_df['epoch'], history_df['train_dice_at_0_5'], label='train')
axes[1].plot(history_df['epoch'], history_df['val_dice_at_0_5'], label='val')
axes[1].set_title('Dice at threshold 0.5')
axes[1].legend()

axes[2].plot(history_df['epoch'], history_df['train_iou_at_0_5'], label='train')
axes[2].plot(history_df['epoch'], history_df['val_iou_at_0_5'], label='val')
axes[2].set_title('IoU at threshold 0.5')
axes[2].legend()

plt.tight_layout()
plt.show()

## 10. Tune Prediction Threshold on Validation Set

In [ ]:
def collect_probs_and_masks(model, loader):
    model.eval()
    all_probs = []
    all_masks = []
    with torch.no_grad():
        for images, masks, _ in tqdm(loader, desc='Collecting predictions'):
            images = images.to(device, non_blocking=True)
            logits = model(images)
            probs = torch.sigmoid(logits).cpu()
            all_probs.append(probs)
            all_masks.append(masks.cpu())
    return torch.cat(all_probs, dim=0), torch.cat(all_masks, dim=0)


best_checkpoint = torch.load(BEST_MODEL_PATH, map_location=device)
model.load_state_dict(best_checkpoint['model_state_dict'])
model.eval()

val_probs, val_masks = collect_probs_and_masks(model, val_loader)
threshold_rows = []
for threshold in np.arange(0.10, 0.91, 0.05):
    threshold_rows.append({
        'threshold': float(threshold),
        'dice': dice_score_from_probs(val_probs, val_masks, threshold=float(threshold)),
        'iou': iou_score_from_probs(val_probs, val_masks, threshold=float(threshold)),
    })

threshold_df = pd.DataFrame(threshold_rows)
BEST_THRESHOLD = float(threshold_df.sort_values('dice', ascending=False).iloc[0]['threshold'])
BEST_VAL_DICE_TUNED = float(threshold_df.sort_values('dice', ascending=False).iloc[0]['dice'])
BEST_VAL_IOU_TUNED = float(threshold_df.sort_values('dice', ascending=False).iloc[0]['iou'])

display(threshold_df.sort_values('dice', ascending=False).head(10))

plt.figure(figsize=(7, 4))
plt.plot(threshold_df['threshold'], threshold_df['dice'], marker='o', label='Dice')
plt.plot(threshold_df['threshold'], threshold_df['iou'], marker='o', label='IoU')
plt.axvline(BEST_THRESHOLD, color='black', linestyle='--', label=f'best threshold = {BEST_THRESHOLD:.2f}')
plt.xlabel('Threshold')
plt.ylabel('Score')
plt.title('Validation threshold sweep')
plt.legend()
plt.tight_layout()
plt.show()

print('Best threshold:', BEST_THRESHOLD)
print('Best validation Dice:', BEST_VAL_DICE_TUNED)
print('Best validation IoU:', BEST_VAL_IOU_TUNED)

## 11. Evaluate on Held-Out Test Set

In [ ]:
test_loss, test_dice_at_threshold, test_iou_at_threshold = run_epoch(model, test_loader, optimizer=None, threshold=BEST_THRESHOLD)
print({
    'test_loss': test_loss,
    'test_dice_at_best_threshold': test_dice_at_threshold,
    'test_iou_at_best_threshold': test_iou_at_threshold,
    'threshold': BEST_THRESHOLD,
})

## 12. Visual Prediction Quality Check

In [ ]:
def clean_mask(mask_binary, min_area_px=120):
    mask_binary = np.asarray(mask_binary).astype(bool)
    mask_binary = ndi.binary_opening(mask_binary, structure=np.ones((3, 3)))
    mask_binary = ndi.binary_closing(mask_binary, structure=np.ones((5, 5)))
    labels, count = ndi.label(mask_binary)
    if count == 0:
        return mask_binary.astype(bool)
    areas = ndi.sum(mask_binary, labels, index=np.arange(1, count + 1))
    largest = int(np.argmax(areas)) + 1
    if areas[largest - 1] < min_area_px:
        return np.zeros_like(mask_binary, dtype=bool)
    return labels == largest


def show_predictions(model, loader, threshold, num_samples=8, save_path=None):
    model.eval()
    shown = 0
    with torch.no_grad():
        for images, masks, names in loader:
            images_device = images.to(device, non_blocking=True)
            probs = torch.sigmoid(model(images_device)).cpu()
            for index in range(images.size(0)):
                if shown >= num_samples:
                    return
                image = denormalize_image(images[index]).permute(1, 2, 0).numpy()
                gt = masks[index, 0].numpy() > 0.5
                prob = probs[index, 0].numpy()
                pred = clean_mask(prob >= threshold)

                tp = gt & pred
                fp = (~gt) & pred
                fn = gt & (~pred)
                error_overlay = np.zeros((*gt.shape, 4), dtype=float)
                error_overlay[tp] = [0.1, 0.8, 0.25, 0.45]
                error_overlay[fp] = [1.0, 0.05, 0.05, 0.70]
                error_overlay[fn] = [0.05, 0.25, 1.0, 0.70]

                fig, axes = plt.subplots(1, 5, figsize=(18, 4))
                axes[0].imshow(image)
                axes[0].set_title(names[index])
                axes[1].imshow(gt, cmap='gray')
                axes[1].set_title('ground truth')
                axes[2].imshow(prob, cmap='magma', vmin=0, vmax=1)
                axes[2].set_title('probability')
                axes[3].imshow(pred, cmap='gray')
                axes[3].set_title('clean prediction')
                axes[4].imshow(image)
                axes[4].imshow(error_overlay)
                axes[4].set_title('green ok / red extra / blue missed')
                for ax in axes:
                    ax.axis('off')
                plt.tight_layout()
                if save_path and shown == 0:
                    plt.savefig(save_path, bbox_inches='tight', dpi=300)
                plt.show()
                shown += 1


show_predictions(model, test_loader, BEST_THRESHOLD, num_samples=8, save_path=PREDICTION_PANEL_PATH)
print('Saved first prediction panel:', PREDICTION_PANEL_PATH)

## 13. Save Deploy Artifacts

In [ ]:
metadata = {
    'architecture': 'runtime_unet_v1',
    'image_size': list(IMAGE_SIZE),
    'threshold': BEST_THRESHOLD,
    'normalization': {'mean': IMAGE_MEAN, 'std': IMAGE_STD},
    'pos_weight': POS_WEIGHT,
    'foreground_pct': FOREGROUND_PCT,
    'best_epoch': best_epoch,
    'val_dice_at_best_threshold': BEST_VAL_DICE_TUNED,
    'val_iou_at_best_threshold': BEST_VAL_IOU_TUNED,
    'test_dice_at_best_threshold': test_dice_at_threshold,
    'test_iou_at_best_threshold': test_iou_at_threshold,
}

best_checkpoint['threshold'] = BEST_THRESHOLD
best_checkpoint['image_size'] = IMAGE_SIZE
best_checkpoint['normalization'] = {'mean': IMAGE_MEAN, 'std': IMAGE_STD}
best_checkpoint['architecture'] = 'runtime_unet_v1'
best_checkpoint['metadata'] = metadata

torch.save(best_checkpoint, DEPLOY_MODEL_PATH)
torch.save(model.state_dict(), PLAIN_STATE_DICT_PATH)
METADATA_PATH.write_text(json.dumps(metadata, indent=2))

print('Saved deploy checkpoint:', DEPLOY_MODEL_PATH)
print('Saved plain state dict:', PLAIN_STATE_DICT_PATH)
print('Saved metadata:', METADATA_PATH)
print('Saved best checkpoint:', BEST_MODEL_PATH)
print('Saved last checkpoint:', LAST_MODEL_PATH)
print('Saved history:', HISTORY_CSV_PATH)
print('Saved prediction panel:', PREDICTION_PANEL_PATH)

## 14. What to Copy Back Into the Repo

After training, copy this file from Drive:

```text
/content/drive/MyDrive/nail_unet_outputs/nail_unet.pt
```

into your local repo as:

```text
models/nail_unet.pt
```

Do not commit `models/nail_unet.pt`. It is ignored by `.gitignore` because it is a trained artifact.

If you choose to use the plain state dict instead, copy:

```text
nail_unet_pi_state_dict.pt
```

and rename it to:

```text
models/nail_unet.pt
```

The deploy checkpoint is preferred because it also stores the tuned threshold and image size, which `nailbot/vision.py` can now read automatically.